# EndoScan AI — Dataset Loader
**Run each cell in order. Do not skip.**

In [14]:
# ── CELL 1: Install dependencies ──────────────────────────────
!pip install datasets transformers torch pandas huggingface_hub -q

In [15]:
# ── CELL 2: Imports ───────────────────────────────────────────
import json
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, pipeline
from huggingface_hub import snapshot_download

results = {}
print('Imports ready')

Imports ready


In [16]:
# ── CELL 3: MTSamples (medical transcription notes) ───────────
mt = pd.read_csv(
    'https://raw.githubusercontent.com/singla007/MTSamples/master/mtsamples.csv'
)
gyn = mt[
    mt['medical_specialty'].str.contains(
        'Obstetrics|Gynecology|Urology', case=False, na=False
    )
]
results['MTSamples'] = {
    'status': ' Loaded',
    'total_records': len(mt),
    'gynaecology_records': len(gyn),
    'columns': mt.columns.tolist()
}
print(f'MTSamples: {len(mt):,} total records')
print(f' Gynaecology subset: {len(gyn):,} records')
print(f'   Columns: {mt.columns.tolist()}')
print(f'\nSample gynaecology note:')
print(gyn['transcription'].iloc[0][:300])

MTSamples: 4,999 total records
 Gynaecology subset: 541 records
   Columns: ['description', 'medical_specialty', 'sample_name', 'transcription', 'keywords']

Sample gynaecology note:
CC:, Confusion and slurred speech.,HX , (primarily obtained from boyfriend): This 31 y/o RHF experienced a "flu-like illness 6-8 weeks prior to presentation. 3-4 weeks prior to presentation, she was found "passed out" in bed, and when awoken appeared confused, and lethargic. She apparently recovered


In [17]:
# ── CELL 4: NCBI Disease NER Dataset ─────────────────────────
ncbi = load_dataset('rjac/biobert-ner-diseases-dataset')
results['NCBI NER'] = {
    'status': ' Loaded',
    'train_records': len(ncbi['train']),
    'test_records': len(ncbi['test']),
    'columns': ncbi['train'].column_names
}
print(f' NCBI NER: {len(ncbi["train"]):,} train | {len(ncbi["test"]):,} test')
print(f'   Columns: {ncbi["train"].column_names}')
print(f'\nSample record:')
print(ncbi['train'][0])

 NCBI NER: 15,488 train | 5,737 test
   Columns: ['tokens', 'tags', 'sentence_id']

Sample record:
{'tokens': ['Selegiline', '-', 'induced', 'postural', 'hypotension', 'in', 'Parkinson', "'", 's', 'disease', ':', 'a', 'longitudinal', 'study', 'on', 'the', 'effects', 'of', 'drug', 'withdrawal', '.'], 'tags': [0, 0, 0, 1, 2, 0, 1, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'sentence_id': 'BC5CDR-0'}


In [18]:
# ── CELL 5: GLENDA (endometriosis laparoscopy images) ────────
glenda_path = snapshot_download(
    repo_id='MFreidank/glenda',
    repo_type='dataset'
)
results['GLENDA'] = {
    'status': ' Downloaded',
    'local_path': glenda_path
}
print(f' GLENDA downloaded')
print(f'   Local path: {glenda_path}')

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

 GLENDA downloaded
   Local path: /home/joy/.cache/huggingface/hub/datasets--MFreidank--glenda/snapshots/e9195188dbd6939e10785e3ed9ee1c513731c9bc


In [19]:
# ── CELL 6: PubMedBERT (NLP base model) ──────────────────────
model_name = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
tokenizer = AutoTokenizer.from_pretrained(model_name)

test_text = 'painful periods and pelvic pain during menstruation'
tokens = tokenizer.convert_ids_to_tokens(
    tokenizer(test_text)['input_ids']
)
results['PubMedBERT'] = {
    'status': ' Loaded',
    'vocab_size': tokenizer.vocab_size
}
print(f' PubMedBERT loaded')
print(f'   Vocab size: {tokenizer.vocab_size:,}')
print(f'   Test tokens: {tokens}')

 PubMedBERT loaded
   Vocab size: 30,522
   Test tokens: ['[CLS]', 'painful', 'periods', 'and', 'pelvic', 'pain', 'during', 'menstr', '##uation', '[SEP]']


In [20]:
# ── CELL 7: Biomedical NER model (live demo) ──────────────────
ner = pipeline(
    'ner',
    model='d4data/biomedical-ner-all',
    aggregation_strategy='simple'
)

test_texts = [
    'I have had painful periods since I was 15',
    'pain during sex and painful bowel movements',
    'heavy menstrual bleeding and chronic fatigue',
    'I cannot get pregnant after trying for two years',
    'pain shoots down my leg and I faint during my period'
]

print(' Biomedical NER — live results on endometriosis symptoms:')
print('=' * 60)
for text in test_texts:
    entities = ner(text)
    print(f'\nInput: "{text}"')
    for e in entities:
        print(f'  {e["word"]:35} → {e["entity_group"]}')

results['Biomedical NER'] = {
    'status': '✅ Loaded',
    'model': 'd4data/biomedical-ner-all'
}

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

 Biomedical NER — live results on endometriosis symptoms:

Input: "I have had painful periods since I was 15"
  painful                             → Sign_symptom
  periods                             → Detailed_description
  i was                               → Duration
  15                                  → Date

Input: "pain during sex and painful bowel movements"
  pain                                → Sign_symptom
  sex                                 → Activity
  painful                             → Sign_symptom
  bow                                 → Diagnostic_procedure
  ##el                                → Sign_symptom
  movements                           → Diagnostic_procedure

Input: "heavy menstrual bleeding and chronic fatigue"
  heavy                               → Severity
  menstrual                           → Biological_structure
  bleeding                            → Sign_symptom
  chronic                             → Detailed_description
  fatigue          

In [21]:
# ── CELL 8: Final summary + delegation ───────────────────────
print('=' * 60)
print('ENDOSCAN AI — DATASET SUMMARY')
print('=' * 60)
for name, info in results.items():
    print(f'\n{name}: {info["status"]}')
    for k, v in info.items():
        if k != 'status':
            print(f'  {k}: {v}')

with open('dataset_summary.json', 'w') as f:
    json.dump(results, f, indent=2, default=str)

print('\n' + '=' * 60)
print('TASK DELEGATION')
print('=' * 60)
print('''
PERSON 1 & 2 — NLP Pipeline
  Datasets : MTSamples (541 gynaecology notes)
             NCBI NER (15,488 BIO-tagged records)
  Model    : PubMedBERT + d4data/biomedical-ner-all
  Tasks    : 1. Filter MTSamples to gynaecology only
             2. Test NER on symptom_aliases.json
             3. Build ESI scoring algorithm from symptoms.json
             4. Output: assess_symptoms(text) → ESI score

PERSON 3 & 4 — Computer Vision Pipeline
  Datasets : GLENDA (downloaded — laparoscopy images)
             MMOTU (download from figshare.com — ultrasound)
  Model    : EfficientNet-B2 (google/efficientnet-b2)
  Tasks    : 1. Download MMOTU from Figshare today
             2. Run EfficientNet-B2 on GLENDA images
             3. Write plain-language description templates
             4. Output: describe_scan(image) → plain text

PERSON 5 — Data Compilation (Group Leader)
  Tasks    : 1. Finish 400-row symptom JSON
             2. Add Kenyan English aliases to symptom_aliases.json
             3. Split into train (320) / val (80)
             4. Hand to NLP team by Wednesday
''')
print(' Summary saved to dataset_summary.json')

ENDOSCAN AI — DATASET SUMMARY

MTSamples:  Loaded
  total_records: 4999
  gynaecology_records: 541
  columns: ['description', 'medical_specialty', 'sample_name', 'transcription', 'keywords']

NCBI NER:  Loaded
  train_records: 15488
  test_records: 5737
  columns: ['tokens', 'tags', 'sentence_id']

GLENDA:  Downloaded
  local_path: /home/joy/.cache/huggingface/hub/datasets--MFreidank--glenda/snapshots/e9195188dbd6939e10785e3ed9ee1c513731c9bc

PubMedBERT:  Loaded
  vocab_size: 30522

Biomedical NER: ✅ Loaded
  model: d4data/biomedical-ner-all

TASK DELEGATION

PERSON 1 & 2 — NLP Pipeline
  Datasets : MTSamples (541 gynaecology notes)
             NCBI NER (15,488 BIO-tagged records)
  Model    : PubMedBERT + d4data/biomedical-ner-all
  Tasks    : 1. Filter MTSamples to gynaecology only
             2. Test NER on symptom_aliases.json
             3. Build ESI scoring algorithm from symptoms.json
             4. Output: assess_symptoms(text) → ESI score

PERSON 3 & 4 — Computer Vision Pi

In [ ]:
import json
import os
from PIL import Image

glenda_path = "/home/joy/Documents/DataScience/Module_6_Capstone/Glenda_v1.5_classes"

# Load annotations
with open(f"{glenda_path}/coco.json") as f:
    coco = json.load(f)

# Load labels
with open(f"{glenda_path}/labels.txt") as f:
    labels = [line.strip() for line in f.readlines()]

# Count images
frames_path = f"{glenda_path}/frames"
images = os.listdir(frames_path)

print(f" GLENDA loaded")
print(f"   Images:      {len(images):,}")
print(f"   Labels:      {labels}")
print(f"   Annotations: {len(coco['annotations']):,}")
print(f"   Categories:  {[c['name'] for c in coco['categories']]}")

# Load sample image
sample = Image.open(f"{frames_path}/{images[0]}")
print(f"   Sample size: {sample.size}")

✅ GLENDA loaded
   Images:      373
   Labels:      ['background', '6.1.1.1_Endo-Peritoneum', '6.1.1.2_Endo-Ovar', '6.1.1.3_Endo-TIE', '6.1.1.4_Endo-Uterus']
   Annotations: 628
   Categories:  ['6.1.1.1_Endo-Peritoneum', '6.1.1.2_Endo-Ovar', '6.1.1.3_Endo-TIE', '6.1.1.4_Endo-Uterus']
   Sample size: (640, 360)
